# GemmaFit v3 LiteRT-LM Conversion Only

Use this notebook after v3 training has already produced the merged HF/safetensors export.

This notebook does not train, does not import Unsloth, and does not use GGUF. It converts:

```text
/content/drive/MyDrive/GemmaFit_train/trained_outputs/merged_fp16/unsloth_gemma-4-E4B-it_gemmafit_v3_evidence_router_merged_fp16
```

to:

```text
/content/drive/MyDrive/GemmaFit_train/gemmafit-v3-evidence-router.litertlm
```

Then it packages a Windows-download bundle:

```text
/content/drive/MyDrive/GemmaFit_train/gemmafit-v3-evidence-router-local-artifacts.zip
```

Important: this notebook uses `--prefill_lengths=[256]`. Do not change it to `--prefill_lengths 256`; the current Fire CLI parses that as an int and fails with `TypeError: 'int' object is not iterable`.


## 0. Mount Drive and inspect artifacts

Run this first. If `Merged HF export` is missing, go back to the training notebook and run merge/export. If `.litertlm` already exists, skip export and run the bundle cell.


In [ ]:
from google.colab import drive
import glob, json, os, re, time
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

WORKDIR = '/content/drive/MyDrive/GemmaFit_train'
TRAINED_OUTPUT_DIR = f'{WORKDIR}/trained_outputs'
RUN_SUFFIX = 'gemmafit_v3_evidence_router'
ARTIFACT_PREFIX = 'gemmafit-v3-evidence-router'

LITERT_EXPORT_DIR = f'{WORKDIR}/litert_export_v3'
LITERTLM_PATH = f'{WORKDIR}/{ARTIFACT_PREFIX}.litertlm'
BUNDLE_ZIP = f'{WORKDIR}/{ARTIFACT_PREFIX}-local-artifacts.zip'
EXPORT_LOG = f'{WORKDIR}/{ARTIFACT_PREFIX}-litert-export.log'

# Prefer the successful v3 evidence-router run if present.
DEFAULT_RUN_NAME = 'unsloth_gemma-4-E4B-it_gemmafit_v3_evidence_router'
DEFAULT_MERGED_PATH = f'{TRAINED_OUTPUT_DIR}/merged_fp16/{DEFAULT_RUN_NAME}_merged_fp16'


def _mtime(path):
    try:
        return os.path.getmtime(path)
    except OSError:
        return 0


def _has_nonzero_safetensors(path):
    files = glob.glob(f'{path}/**/*.safetensors', recursive=True)
    return bool(files) and all(os.path.getsize(f) > 0 for f in files if os.path.isfile(f))

merged_candidates = []
if os.path.isdir(DEFAULT_MERGED_PATH):
    merged_candidates.append(DEFAULT_MERGED_PATH)
merged_candidates += [
    p for p in glob.glob(f'{TRAINED_OUTPUT_DIR}/merged_fp16/*_{RUN_SUFFIX}_merged_fp16')
    if p != DEFAULT_MERGED_PATH and os.path.isdir(p)
]
merged_candidates = [p for p in merged_candidates if _has_nonzero_safetensors(p)]
merged_candidates = sorted(merged_candidates, key=_mtime, reverse=True)

if not merged_candidates:
    raise FileNotFoundError(
        'No valid merged HF export found. Expected: '        f'{DEFAULT_MERGED_PATH}. Run the training notebook merge phase first.'
    )

MERGED_PATH = merged_candidates[0]
RUN_NAME = Path(MERGED_PATH).name.removesuffix('_merged_fp16')
DONE_MARKER = f'{TRAINED_OUTPUT_DIR}/TRAINING_DONE_{RUN_NAME}.json'
RUN_STATE = f'{TRAINED_OUTPUT_DIR}/RUN_STATE_{RUN_NAME}.json'
RUN_EVENTS = f'{TRAINED_OUTPUT_DIR}/RUN_EVENTS_{RUN_NAME}.jsonl'
DISCONNECT_POINTS = f'{TRAINED_OUTPUT_DIR}/DISCONNECT_POINTS_{RUN_NAME}.jsonl'
CKPT_DIR = f'{TRAINED_OUTPUT_DIR}/checkpoints/{RUN_NAME}'

print('=== GemmaFit v3 LiteRT conversion inputs ===')
print('Run name         :', RUN_NAME)
print('Merged HF export :', MERGED_PATH)
print('Final .litertlm  :', LITERTLM_PATH, 'FOUND' if os.path.exists(LITERTLM_PATH) else 'missing')
print('Export dir       :', LITERT_EXPORT_DIR)
print('Export log       :', EXPORT_LOG)
print('Bundle zip       :', BUNDLE_ZIP, 'FOUND' if os.path.exists(BUNDLE_ZIP) else 'missing')
print('Done marker      :', DONE_MARKER, 'FOUND' if os.path.exists(DONE_MARKER) else 'missing')

if os.path.exists(DONE_MARKER):
    try:
        done = json.load(open(DONE_MARKER, encoding='utf-8'))
        print('Done status      :', done.get('status'), '/', done.get('conversion_status'))
    except Exception as exc:
        print('Done marker read warning:', exc)


## 1. Install LiteRT conversion dependencies

Run this in a conversion-only runtime. This cell upgrades LiteRT dependencies and PyTorch nightly, then intentionally restarts the Colab runtime so the new torch is actually loaded. After the restart, rerun cells 0 and 2.


In [ ]:
import os, signal, subprocess, sys

INSTALL_LITERT_DEPS = True
# Current litert-torch-nightly imports require torch >= 2.11.0 plus a matching
# torchao package exposing torchao.quantization.pt2e.quantize_pt2e.
UPGRADE_TORCH_NIGHTLY = True
FORCE_RUNTIME_RESTART_AFTER_INSTALL = True

if INSTALL_LITERT_DEPS:
    cmd = [
        sys.executable, '-m', 'pip', 'install', '-U', '--pre',
        'litert-torch-nightly', 'litert-lm', 'sentencepiece', 'protobuf',
        'transformers', 'huggingface_hub', 'accelerate', 'safetensors',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

if UPGRADE_TORCH_NIGHTLY:
    cmd = [
        sys.executable, '-m', 'pip', 'install', '-U', '--pre',
        'torch', 'torchao',
        '--index-url', 'https://download.pytorch.org/whl/nightly/cu128',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

    # Some Colab images keep an older torchao from PyPI. Force a second pass if
    # the nightly index did not replace it cleanly.
    cmd = [sys.executable, '-m', 'pip', 'install', '-U', '--pre', '--force-reinstall', 'torchao']
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

print('\nInstall finished. Colab must restart so litert_torch sees upgraded torch/torchao.')
print('After restart: rerun cell 0, skip cell 1, then run cell 2.')
print('Do not import Unsloth in this conversion-only runtime.')

if FORCE_RUNTIME_RESTART_AFTER_INSTALL:
    os.kill(os.getpid(), signal.SIGKILL)


## 2. Verify exporter after restart

This tolerates Fire's non-zero help return code as long as the help output contains the expected flags.


In [ ]:
import importlib.util, re, subprocess, sys

print('Python:', sys.version)
torch_version = None
try:
    import torch
    torch_version = torch.__version__
    print('torch:', torch_version)
    print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as exc:
    print('torch import warning:', exc)

try:
    import torchao
    print('torchao:', getattr(torchao, '__version__', 'unknown'))
except Exception as exc:
    raise RuntimeError(
        'torchao is not importable. Rerun install cell 1, let it restart runtime, then rerun cells 0 and 2.'
    ) from exc

if importlib.util.find_spec('torchao.quantization.pt2e.quantize_pt2e') is None:
    raise RuntimeError(
        'torchao is present but missing torchao.quantization.pt2e.quantize_pt2e. '
        'Rerun install cell 1 from the latest notebook so torchao is upgraded/reinstalled, '
        'let Colab restart, then rerun cells 0 and 2.'
    )

def _version_tuple(version):
    match = re.match(r'(\d+)\.(\d+)', version or '')
    return tuple(map(int, match.groups())) if match else (0, 0)

if torch_version and _version_tuple(torch_version) < (2, 11):
    raise RuntimeError(
        f'torch {torch_version} is too old for litert-torch-nightly. '
        'Rerun install cell 1, let it restart runtime, then rerun cells 0 and 2.'
    )

help_cmd = [sys.executable, '-m', 'litert_torch.generative.export_hf', '--help']
help_proc = subprocess.run(help_cmd, text=True, capture_output=True, check=False)
help_text = (help_proc.stdout or '') + (help_proc.stderr or '')
print('export_hf --help returncode:', help_proc.returncode)
print(help_text[:3000])

if 'Please upgrade to torch >= 2.11.0' in help_text:
    raise RuntimeError(
        'litert_torch import still sees an old torch. Restart runtime after upgrading torch, then rerun cells 0 and 2.'
    )
if "No module named 'torchao.quantization.pt2e.quantize_pt2e'" in help_text:
    raise RuntimeError(
        'litert_torch import still sees an incompatible torchao. Rerun install cell 1 from the latest notebook, '
        'let Colab restart, then rerun cells 0 and 2.'
    )

help_looks_valid = (
    'SYNOPSIS' in help_text
    and 'POSITIONAL ARGUMENTS' in help_text
    and '--quantization_recipe' in help_text
    and '--prefill_lengths' in help_text
)
if help_proc.returncode != 0 and help_looks_valid:
    print('Valid Fire help with non-zero return code; continuing is OK.')
elif help_proc.returncode != 0:
    raise RuntimeError('LiteRT exporter is not importable. Restart runtime after pip install, then rerun this cell.')


## 3. Convert merged HF to `.litertlm`

This cell runs one clean export command first:

```text
--prefill_lengths=[256]
--quantization_recipe=dynamic_wi4_afp32
```

It does not run the broken `--prefill_lengths 256` form.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

FORCE_EXPORT = True
PREFILL_LITERAL = '[256]'
CACHE_LENGTH = '2048'
QUANTIZATION_RECIPE = 'dynamic_wi4_afp32'

if os.path.exists(LITERTLM_PATH) and not FORCE_EXPORT:
    print('Existing .litertlm found and FORCE_EXPORT=False:', LITERTLM_PATH)
else:
    shutil.rmtree(LITERT_EXPORT_DIR, ignore_errors=True)
    Path(LITERT_EXPORT_DIR).mkdir(parents=True, exist_ok=True)

    commands = [
        [
            sys.executable, '-m', 'litert_torch.generative.export_hf',
            MERGED_PATH, LITERT_EXPORT_DIR,
            f'--prefill_lengths={PREFILL_LITERAL}',
            f'--cache_length={CACHE_LENGTH}',
            f'--quantization_recipe={QUANTIZATION_RECIPE}',
            '--externalize_embedder',
            '--use_jinja_template',
        ],
        [
            sys.executable, '-m', 'litert_torch.generative.export_hf',
            MERGED_PATH, LITERT_EXPORT_DIR,
            '-p', PREFILL_LITERAL,
            f'--cache_length={CACHE_LENGTH}',
            f'--quantization_recipe={QUANTIZATION_RECIPE}',
            '--externalize_embedder',
            '--use_jinja_template',
        ],
    ]

    last_output = ''
    with open(EXPORT_LOG, 'w', encoding='utf-8') as log:
        for attempt, cmd in enumerate(commands, 1):
            if attempt > 1:
                shutil.rmtree(LITERT_EXPORT_DIR, ignore_errors=True)
                Path(LITERT_EXPORT_DIR).mkdir(parents=True, exist_ok=True)
            print('\n=== LiteRT export attempt', attempt, '===')
            print(' '.join(cmd))
            log.write('\n=== LiteRT export attempt %d ===\n' % attempt)
            log.write(' '.join(cmd) + '\n')
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            chunks = []
            for line in proc.stdout:
                print(line, end='')
                log.write(line)
                chunks.append(line)
            rc = proc.wait()
            last_output = ''.join(chunks)
            log.write(f'\nreturncode={rc}\n')
            log.flush()
            if rc == 0:
                break
            if "TypeError: 'int' object is not iterable" in last_output:
                print('Detected int prefill parse failure; trying list-literal fallback.')
                continue
            raise RuntimeError(f'LiteRT export failed with return code {rc}. See {EXPORT_LOG}')
        else:
            raise RuntimeError(f'LiteRT export failed after all attempts. See {EXPORT_LOG}')

    candidates = sorted(Path(LITERT_EXPORT_DIR).rglob('*.litertlm'), key=lambda x: x.stat().st_size, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No .litertlm artifact found under {LITERT_EXPORT_DIR}. See {EXPORT_LOG}')
    Path(LITERTLM_PATH).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(candidates[0], LITERTLM_PATH)
    print('\nCopied LiteRT-LM artifact:', candidates[0], '->', LITERTLM_PATH)
    print('Size bytes:', os.path.getsize(LITERTLM_PATH))

# Update training done marker without claiming smoke has passed.
done = {}
if os.path.exists(DONE_MARKER):
    with open(DONE_MARKER, encoding='utf-8') as f:
        done = json.load(f)
done.update({
    'version': 'v3_evidence_router',
    'run_name': RUN_NAME,
    'run_suffix': RUN_SUFFIX,
    'last_completed_phase': '10_litert_conversion',
    'merged_hf_path': MERGED_PATH,
    'litertlm_path': f'models/{ARTIFACT_PREFIX}.litertlm',
    'conversion_status': 'converted_unverified',
    'conversion_log': {
        'source_hf': MERGED_PATH,
        'export_dir': LITERT_EXPORT_DIR,
        'colab_output': LITERTLM_PATH,
        'export_log': EXPORT_LOG,
        'quantization_recipe': QUANTIZATION_RECIPE,
        'prefill_lengths': [256],
        'cache_length': int(CACHE_LENGTH),
        'note': 'Smoke test still required before ready_for_android.',
    },
    'tool_call_eval': 'finetune/metrics/tool_call_eval_v3.json',
})
done.setdefault('finished_at', time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
Path(DONE_MARKER).parent.mkdir(parents=True, exist_ok=True)
with open(DONE_MARKER, 'w', encoding='utf-8') as f:
    json.dump(done, f, indent=2, ensure_ascii=False)
print('Done marker updated:', DONE_MARKER)


## 4. Package local download bundle

Download the zip from Drive and run the local finalize command in `D:\GemmaFit`.


In [ ]:
import os, shutil, zipfile
from pathlib import Path

BUNDLE_ROOT = Path('/content/gemmafit_v3_litert_bundle')
if not os.path.exists(LITERTLM_PATH):
    raise FileNotFoundError(f'LiteRT-LM artifact not found: {LITERTLM_PATH}. Run the export cell first.')

shutil.rmtree(BUNDLE_ROOT, ignore_errors=True)

copies = {
    LITERTLM_PATH: f'models/{ARTIFACT_PREFIX}.litertlm',
    DONE_MARKER: 'finetune/metrics/training_done_v3.json',
    RUN_STATE: 'finetune/metrics/run_state_v3.json',
    RUN_EVENTS: 'finetune/metrics/run_events_v3.jsonl',
    DISCONNECT_POINTS: 'finetune/metrics/disconnect_points_v3.jsonl',
    EXPORT_LOG: f'finetune/metrics/{ARTIFACT_PREFIX}-litert-export.log',
}
trainer_states = sorted(Path(CKPT_DIR).glob('checkpoint-*/trainer_state.json'), key=lambda p: p.stat().st_mtime, reverse=True)
if trainer_states:
    copies[str(trainer_states[0])] = 'finetune/metrics/trainer_state_v3.json'

for src, rel in copies.items():
    if not src or not os.path.exists(src):
        print('Skipping missing artifact:', src)
        continue
    dst = BUNDLE_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

with zipfile.ZipFile(BUNDLE_ZIP, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    for path in BUNDLE_ROOT.rglob('*'):
        if path.is_file():
            zf.write(path, path.relative_to(BUNDLE_ROOT).as_posix())

print('Download bundle:', BUNDLE_ZIP)
print('Bundle size bytes:', os.path.getsize(BUNDLE_ZIP))
print('\nWindows local finalize command:')
print(r'cd /d D:\GemmaFit')
print(r'python finetune\prepare_litert_artifact.py --source-bundle "C:\path\to\gemmafit-v3-evidence-router-local-artifacts.zip" --run-smoke')


## 5. Optional: push model to Pixel after local finalize

After local smoke passes and `models/gemmafit-v3-evidence-router.litertlm` exists on Windows:

```powershell
adb push models/gemmafit-v3-evidence-router.litertlm /sdcard/Android/data/com.gemmafit/files/gemmafit-v3-evidence-router.litertlm
adb shell run-as com.gemmafit ls -lh files
```

If `run-as` is not available on your build, use the app's debug model path screen or `adb shell ls -lh /sdcard/Android/data/com.gemmafit/files/`.
